### Installiamo `gurobipy`, documentazione @ https://www.gurobi.com/documentation/9.5/refman/py_python_api_details.html#sec:Python-details

In [1]:
!pip3 install gurobipy

# Importiamo i pacchetti necessari

In [2]:
import gurobipy as gp           # Libreria principale
from gurobipy import GRB        # Costanti (MaXIMIZE, INFINITY, ecc..)
import numpy as np              # Per array e operazioni matematiche

# Vogliamo definire e risolvere il seguente problema

\begin{align}
  \text{maximize}\ &250x_1 + 230x_2 + 110x_3 + 350x_4 + 110x_5\\
                  &s.t.\\
                  &0\le\ x_i \lt \infty\\
                  &2x_1 + 1.5x_2 + 0.5x_3 + 2.5x_4 + 0.7x_5\leq 100.\\
                  &0.5x_1 + 0.25x_2 + 0.25x_3 + x_4 + 0.3x_5\leq 50\\
                  &x_1 + x_2 \geq 10\\
                  &x_3 = x_5\\
\end{align}

Creiamo il modello

In [3]:
model = gp.Model()

Restricted license - for non-production use only - expires 2027-11-29


Aggiungiamo le variabili con i rispettivi limiti

In [4]:
# Numero di variabili del problema (x1, x2, x3, x4, x5)
n_vars = 5

# Creo un vettore di variabili decisionali: x = (x1, x2, x3, x4, x5)
# addMVar → crea più variabili tutte insieme (forma vettoriale)

x = model.addMVar(
    n_vars,                 # Numero di variabili
    lb=[0] * n_vars,        # Lower bound: tutte >= 0
    ub=GRB.INFINITY         # Upper bound: infinito (nessun limite superiore)
)

# Aggiorna il modello dopo aver aggiunto le variabili
model.update()

# Stampo le variabili

print(f"Model vars:{model.getVars()}")        # stampa tutte le variabili
print(f"Num vars: {len(model.getVars())}")    # Numero variabili
print(f"Var C1: {model.getVarByName('C0')}")  # Prende la prima variabile (x1)

Model vars:[<gurobi.Var C0>, <gurobi.Var C1>, <gurobi.Var C2>, <gurobi.Var C3>, <gurobi.Var C4>]
Num vars: 5
Var C1: <gurobi.Var C0>


Aggiungiamo i vincoli del problema

In [5]:
# Matrice dei coefficenti dei vincoli

A =  np.array([[2., 1.5, 0.5, 2.5, 0.7],        # Vincolo 1
               [0.5, 0.25, 0.25, 1, 0.3],       # Vincolo 2
               [1, 1, 0, 0, 0],                 # Vincolo 3
               [0, 0, 1, 0, -1]])               # Vincolo 4

b = np.array([100, 50, 10, 0])                  # Termini noti (lato destro)

ct = model.addConstr(A[:2]@x <= b[:2])          # Primi 2 vincoli (<=)
ct2 = model.addConstr(A[-2]@x >= b[-2])         # Terzo vincolo (>=)
ct3 = model.addConstr(A[-1]@x == b[-1])         # Quarto vincolo (=)

model.update()                                  # Aggiorno il modello

Definiamo la funzione obiettivo

In [6]:
# Coefficenti della funzione obbiettivo
obj_coefs = np.array([250, 230, 110, 350, 110])    

# Max profitto
model.setObjective(obj_coefs @ x, GRB.MAXIMIZE)

Risolviamo il problema

In [7]:
model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: Intel(R) Core(TM) i7-6700HQ CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4 rows, 5 columns and 14 nonzeros (Max)
Model fingerprint: 0x98c1760a
Model has 5 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-01, 2e+00]
  Objective range  [1e+02, 4e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+02]

Presolve removed 1 rows and 1 columns
Presolve time: 0.02s
Presolved: 3 rows, 4 columns, 10 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.8333333e+04   2.500000e+00   0.000000e+00      0s
       1    1.7883333e+04   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.03 seconds (0.00 work units)
Optimal objective  1.788333333e+04


Stampiamo per ogni variabile il valore nella soluzione ottima



In [8]:
for v in model.getVars():
  print('%s %g' % (v.VarName, v.X))      # Nome variabile + valore

print('Obj: %g' % model.ObjVal)          # Valore ottimo funzione obbiettivo   

C0 0
C1 10
C2 70.8333
C3 0
C4 70.8333
Obj: 17883.3


# Proviamo a risolvere il problema duale

In [9]:
print ("\nValori duali:")

for c in model.getConstrs():            # Ciclo su tutti i vincoli
    print(f"{c.ConstrName}: {c.Pi}")    # Pi = valore duale


Valori duali:
R0: 183.33333333333334
R1: 0.0
R2: -45.0
R3: 18.333333333333343


# E' possibile accedere ai vincoli ed alla funzione obiettivo

In [15]:
for c in model.getConstrs():          # Ciclo sui vincoli del modello primale
    print(model.getRow(c))            # Stampo la parte sinistra del vincolo

model.getObjective()                  # Mostro la funzione obiettivo del modello

2.0 C0 + 1.5 C1 + 0.5 C2 + 2.5 C3 + 0.7 C4
0.5 C0 + 0.25 C1 + 0.25 C2 + C3 + 0.3 C4
C0 + C1
C2 + -1.0 C4


<gurobi.LinExpr: 250.0 C0 + 230.0 C1 + 110.0 C2 + 350.0 C3 + 110.0 C4>

# Rendiamo il problema primale privo di soluzioni ammissibili

In [11]:
# Creo un nuovo modello
model_inf = gp.Model ()

# Variabili
n_var = 5
x_inf = model_inf.addMVar (n_vars, lb=[0]*n_vars, ub=GRB.INFINITY)
model_inf.update()

# Vincoli originali
model_inf.addConstr(A[:2] @ x_inf <= b[:2])
model_inf.addConstr(A[-2] @ x_inf >= b[-2])
model_inf.addConstr(A[-1] @ x_inf == b[-1])

# Vincoli in contraddizione
model_inf.addConstr(x_inf[0] <= 1)
model_inf.addConstr(x_inf[0] >= 10)

# Funzione obbiettivo
model_inf.setObjective(obj_coefs @ x_inf, GRB.MAXIMIZE)

# Risoluzione
model_inf.optimize()

# Risultato
if model_inf.Status == GRB.INFEASIBLE:
    print("Il modello è infeasible")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: Intel(R) Core(TM) i7-6700HQ CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 6 rows, 5 columns and 16 nonzeros (Max)
Model fingerprint: 0x0220211f
Model has 5 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-01, 2e+00]
  Objective range  [1e+02, 4e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+02]

Presolve time: 0.01s

Solved in 0 iterations and 0.02 seconds (0.00 work units)
Infeasible or unbounded model


# Rendiamo il problema primale illimitato

In [14]:
# Creo un nuovo modello
model_unb = gp.Model()

# Variabile senza limite superiore
x_unb = model_unb.addVar (lb=0)

# Funzione obbiettivo (cresce sempre)
model_unb.setObjective(100 * x_unb, GRB.MAXIMIZE)   

# Risoluzione
model_unb.optimize()

# Controllo
if model_unb.Status == GRB.UNBOUNDED:
    print("Il modello è unbounded")
else:
    print("Status:", model_unb.Status)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: Intel(R) Core(TM) i7-6700HQ CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 0 rows, 1 columns and 0 nonzeros (Max)
Model fingerprint: 0x6db48de3
Model has 1 linear objective coefficients
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  Objective range  [1e+02, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [0e+00, 0e+00]

Presolve time: 0.02s

Solved in 0 iterations and 0.02 seconds (0.00 work units)
Unbounded model
Il modello è unbounded


# Problema del cammino minimo
Dato un grafo $\mathcal{G}$ e due nodi $s,\ t$ vogliamo trovare il cammino che minimiza il costo di andare da $s$ a $t$.

Creiamo un grafo costituito da $200$ nodi

In [ ]:
n_nodes = 200                               # Numero di nodi del grafo

adj_mat = np.random.rand(n_nodes, n_nodes)  # Matrice casuale (pesi archi)
adj_mat[adj_mat > 0.02] = 0                 # Rende il grafo sparso (pochi archi)
adj_mat *= 1000                             # Scala i costi
adj_mat = np.round(adj_mat)                 # Arrotonda i pesi
np.fill_diagonal(adj_mat, 0)                # Niente archi da nodo a sé stesso

Definiamo il nodo di partenza ed il nodo di arrivo

In [25]:
s = 0       # Nodo di partenza
t = 197     # Nodo di arrivo

## Risolviamo il problema con Gurobi

Se il costo di ogni arco e' positivo, il problema si puo' formulare nel seguente modo:
\begin{align}
  \text{minimize} &\sum_{ij} c_{ij}x_{ij}\\
                  & s.t.\\
                  0 \leq\ &x \leq 1\\
                  \sum_i x_{iv} + b_v &= \sum_j x_{vj},\ \forall v \in \text{Nodes}
\end{align}
dove $b_v$ il flusso in ingresso su ogni nodo ed $x_{ij}$ la quantita' di flusso nell'arco $(i, j)$.

In [26]:
import gurobipy as gp
from gurobipy import GRB

Creiamo il modello

In [27]:
m = gp.Model('shortestPath')

Aggiungiamo le variabili e definiamo la funzione obiettivo

In [30]:
# Trovo gli indici degli archi esistenti (dove peso !=0)
var_idxs = np.nonzero(adj_mat)

# Li trasformo in una lista di tuple (i, j)
var_idxs = [(i, j) for i, j in zip(var_idxs[0], var_idxs[1])]

# Creo un dizionario dei costi per (i, j)
costs = {idx: adj_mat[idx] for idx in var_idxs}

# Creo variabili sugli archi:   - x_ij = 1 → arco usato nel cammino
#                               - x_ij = 0 → arco non usato


arcs = m.addVars(
    var_idxs,              # Archi disponibili
    lb=0,                  # Flusso minimo
    ub=1,                  # Flusso massimo (tipo booleano continuo)
    obj=costs,             # Coefficiente funzione obiettivo
    name="arcs"
)

m.update()

In [29]:
m.getObjective()

<gurobi.LinExpr: 5.0 arcs[0,13] + 18.0 arcs[0,19] + 16.0 arcs[0,43] + 9.0 arcs[0,58] + 5.0 arcs[0,112] + 16.0 arcs[0,123] + 16.0 arcs[1,92] + 19.0 arcs[1,98] + 11.0 arcs[1,171] + 13.0 arcs[2,6] + 15.0 arcs[2,20] + 6.0 arcs[2,52] + 13.0 arcs[2,84] + 14.0 arcs[2,135] + 2.0 arcs[3,31] + 7.0 arcs[4,93] + 4.0 arcs[6,4] + 11.0 arcs[6,115] + 14.0 arcs[6,137] + 6.0 arcs[6,155] + 15.0 arcs[6,161] + 6.0 arcs[6,190] + 8.0 arcs[7,110] + 19.0 arcs[7,147] + 9.0 arcs[8,78] + 18.0 arcs[8,99] + 3.0 arcs[8,102] + 5.0 arcs[8,104] + 11.0 arcs[8,146] + 6.0 arcs[9,8] + arcs[9,22] + 10.0 arcs[9,29] + 15.0 arcs[9,96] + 7.0 arcs[10,150] + 20.0 arcs[10,153] + 3.0 arcs[11,8] + 13.0 arcs[11,16] + 19.0 arcs[11,24] + 7.0 arcs[11,51] + 18.0 arcs[11,115] + 4.0 arcs[11,164] + 18.0 arcs[12,93] + 5.0 arcs[12,139] + 6.0 arcs[13,1] + 9.0 arcs[13,15] + 5.0 arcs[13,90] + 20.0 arcs[13,92] + 19.0 arcs[13,140] + 8.0 arcs[13,170] + arcs[14,24] + 16.0 arcs[14,33] + 13.0 arcs[14,48] + 13.0 arcs[14,54] + 16.0 arcs[14,63] + 11.0 ar

Aggiungiamo i vincoli

In [ ]:
b = np.zeros(n_nodes)     # Vettore flusso

b[s] = 1                  # Nodo sorgente → manda flusso
b[t] = -1                 # Nodo destinazione → riceve flusso

# Vincolo: flusso entrante + b[v] = flusso uscente
# → conservazione del flusso

m.addConstrs(
    (arcs.sum('*', v) + b[v] == arcs.sum(v, '*') for v in range(n_nodes))
)


Risolviamo il problema e stampiamo la soluzione trovata

In [32]:
m.optimize()

print(f"Costo minimo: {m.objVal:.4f}")

# Stampo gli archi usati nel cammino

for v in var_idxs:
    if arcs[v].X > 0.5:   # Se l'arco è stato scelto
        print(f"{v[0]} -> {v[1]} (costo {arcs[v].Obj})")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: Intel(R) Core(TM) i7-6700HQ CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 0 rows, 1594 columns and 0 nonzeros (Min)
Model has 1594 linear objective coefficients
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  Objective range  [1e+00, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [0e+00, 0e+00]


Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00
Costo minimo: 0.0000


# Misc Utili

In [33]:
a = [1,2,3,4]
b = ["a", "b", "c", "d"]

## Itertools
Implementa molte funzioni utili per iterare su collezioni di elementi

In [34]:
import itertools

In [ ]:
print(list(itertools.chain(a, b)))

[1, 2, 3, 4, 'a', 'b', 'c', 'd']


In [ ]:
print(list(itertools.product(a,b)))

[(1, 'a'), (1, 'b'), (1, 'c'), (1, 'd'), (2, 'a'), (2, 'b'), (2, 'c'), (2, 'd'), (3, 'a'), (3, 'b'), (3, 'c'), (3, 'd'), (4, 'a'), (4, 'b'), (4, 'c'), (4, 'd')]


## zip, enumerate

In [ ]:
for a_i, b_i in zip(a, b):
  print(a_i, b_i)

1 a
2 b
3 c
4 d


In [ ]:
for x in zip(a, b):
  print(x)

(1, 'a')
(2, 'b')
(3, 'c')
(4, 'd')


In [ ]:
for i, a_i in enumerate(a):
  print(i, a_i)

0 1
1 2
2 3
3 4


In [ ]:
for x in enumerate(a):
  print(x)

(0, 1)
(1, 2)
(2, 3)
(3, 4)


# Esercizio

Un polo industriale produce televisori utilizzando tre linee di montaggio (A, B e C) per televisori piatti, curvi e monitor ad alte prestazioni. Per stabilire la produzione per il prossimo semestre e’ necessario considerare I seguenti aspetti:

  * Dalla vendita dei tre prodotti l’azienda ricava un profitto pari a 200, 400 e 500 euro rispettivamente.

  * Nel corso dei sei mesi, la linea A potra’ essere impengata per un totale di 150 ore al mese, la linea B per 130 al mese e la linea C per 170 ore al mese. Tuttavia, esclusivamente per il quarto mese di produzione sarà possibile usufruire di ulteriori 20 ore di lavoro sulla linea A e di 10 ore in meno sulla linea C.

  * Produrre un televisore a schermo piatto richiede 18 ore sulla linea A e 7 sulla B, un televisore curvo invece ne richiede 12 sulla linea B e 25 sulla C, mentre un monitor ad alte prestazioni richiede 10 ore di lavorazione sulla linea A, 9 sulla linea B e 14 sulla linea C.

  * In un impianto separato l’azienda esegue verifiche sui prodotti ed ha una capacita’ lavorativa mensile pari a 120 ore. Per eseguire tutte le verifiche necessarie sono richieste 5 ore per un televisore a schermo piatto, 8 per uno curvo e 9 per un monitor.

  * A seguito di analisi di mercato vi sono I seguenti vincoli sulla produzione: nel secondo mese la produzione di monitor deve supereare le 10 unita’, nel terzo e quinto mese la produzione di televisori curvi deve essere almeno pari al 30% di quella di televisori piatti ed infine nel sesto mese di produzione non e’ prevista richiesta di monitor ad alte prestazioni.

  * Temendo uno sciopero dei lavoratori, la direzione considera che nel primo mese di produzione la capacita’ lavorativa dell’impianto di verifica dei prodotti sara’ ridotta al 75%.

  * La produzione complessiva sui 6 mesi di monitor ad alte prestazioni non deve superare le 40 unità.

Sulla base di tali considerazioni, il problema e’ pianificiare la produzione del prossimo semestre con l’obiettivo di massimizzare il profitto dell’azienda.

In [ ]:
#TODO